In [21]:
import sys
!{sys.executable} -m pip install -U pymupdf

In [22]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [23]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params, timeout=60.0)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [24]:
import fitz  # PyMuPDF
import base64

# Constants for production thresholds
MIN_IMAGE_DIMENSION = 100  # Minimum pixels (width or height) to be considered a "real" image

def analyze_pdf_and_route(pdf_path):
    """
    Production-grade PDF router.
    Filters out tiny images (logos, icons, artifacts) before deciding the route.
    """
    doc = fitz.open(pdf_path)
    
    total_text = ""
    significant_images = 0
    ignored_images = 0
    
    # Analyze each page
    for page in doc:
        total_text += page.get_text()
        
        # page.get_images() returns a list of tuples.
        # Format: (xref, smask, width, height, bpc, colorspace, ext, image, filter)
        image_list = page.get_images(full=True)
        
        for img in image_list:
            width = img[2]
            height = img[3]
            
            # Check if the image is large enough to warrant Vision API
            if width >= MIN_IMAGE_DIMENSION and height >= MIN_IMAGE_DIMENSION:
                significant_images += 1
            else:
                ignored_images += 1
        
    print(f"📊 Extraction: {len(total_text)} characters read.")
    print(f"   🖼️  Significant images (>= {MIN_IMAGE_DIMENSION}px): {significant_images}")
    print(f"   🗑️  Ignored tiny images/artifacts: {ignored_images}")

    # Routing Logic based only on significant images
    if significant_images > 0:
        print("🔀 Route: Native Vision API (Significant visual context found)")
        with open(pdf_path, "rb") as f:
            file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")
        
        return {
            "type": "document",
            "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": file_bytes
            },
            "title": pdf_path,
            "citations": {"enabled": True}
        }
    else:
        print("🔀 Route: Text-Only API (No significant images, optimizing cost)")
        return {
            "type": "document",
            "source": {
                "type": "text",
                "media_type": "text/plain",
                "data": total_text
            },
            "title": pdf_path,
            "citations": {"enabled": True}
        }

# 1. Run the router on our document
pdf_files = [
    "Falsifiable Commitment Planning for Self-Correcting Web Agents.pdf",
    "AttriMem- Attribution-Guided Process Feedback for Agent Memory Learning.pdf",
    "Mi-Memory- A Lifecycle Memory Framework for Personal AI.pdf",
    "TRACE-ROUTER- Task-Consistent and Adaptive Online Routing for Agentic AI.pdf",
    "FEATURES_OF_CLAUDE.pdf",
    "earth.pdf"
]

for file_name in pdf_files:
    print(f"\n{'='*50}\n📄 Analyzing file: {file_name}\n{'='*50}")
    
    # 1. Routing logic
    try:
        pdf_document_block = analyze_pdf_and_route(file_name)
    except Exception as e:
        print(f"❌ Error processing file {file_name}: {e}")
        continue
        
    # 2. Create a new message array FOR EACH file inside the loop
    messages = []
    
    # 3. Add the user message payload
    add_user_message(
        messages,
        [
            pdf_document_block,
            {
                "type": "text",
                "text": "Please summarize the main contribution or core idea of this research paper in 3-4 sentences. Always provide exact citations from the document to support your summary."
            }
        ]
    )
    
    # 4. Call Claude API
    print("⏳ Calling Claude API...")
    try:
        response = chat(messages)
        
        print("\n✅ CLAUDE'S RESPONSE:\n")
        for block in response.content:
            if block.type == "text":
                print(block.text)
            
            # Extract and display citations if they exist
            elif block.type == "citations":
                print("\n--- 📌 CITATIONS USED ---")
                for cite in block.citations:
                    snippet = cite.cited_text[:80].replace("\n", " ")
                    print(f"• [{cite.document_title}]: \"{snippet}...\"")
                    
    except Exception as e:
        print(f"❌ API Error for file {file_name}: {e}")




📄 Analyzing file: Falsifiable Commitment Planning for Self-Correcting Web Agents.pdf
📊 Extraction: 72358 characters read.
   🖼️  Significant images (>= 100px): 20
   🗑️  Ignored tiny images/artifacts: 25
🔀 Route: Native Vision API (Significant visual context found)
⏳ Calling Claude API...

✅ CLAUDE'S RESPONSE:

FCPAgent proposes a falsifiable commitment planning framework for robust long-horizon web agents, where each plan step is represented as a Falsifiable Commitment Unit (FCU): a subgoal grounded in a reusable skill, together with confirming evidence, falsifying evidence, and a confidence score.
 
Execution is organized as a plan-test-repair loop: the hybrid commitment testing module checks candidate actions before they modify the browser and checks observations after execution; when evidence falsifies a commitment, scope-aware repair localizes the contradiction to the execution, skill, or planning level and revises the smallest adequate part.
 
On WebArena, FCPAgent achieves a 13